<img src="uva_seal.png">

## Kafka from Scratch with PySpark

### University of Virginia
### DS 7200: Distributed Computing
### Last Updated: November 1, 2026

---

### SOURCES

https://kafka.apache.org/documentation/#design

https://spark.apache.org/docs/latest/api/python/tutorial/sql/python_data_source.html

https://spark.apache.org/docs/latest/streaming/structured-streaming-kafka-integration.html

### ABOUT

In this demo we build a tiny Kafka, **MiniKafka**, using nothing but Python files and PySpark. There is no broker to install. The "broker" is a folder on disk, so you can open every file and see exactly what Kafka stores.

We then plug MiniKafka into Spark using the **Python Data Source API** (new in Spark 4.0). That API asks us to implement the same contract Spark's real Kafka connector implements: *what is the first offset, what is the latest offset, and which records lie between two offsets?* By the end, `spark.readStream.format("minikafka")` behaves like `spark.readStream.format("kafka")`.

| Kafka concept | Real Kafka | MiniKafka |
|---|---|---|
| Broker | JVM server process | a folder: `minikafka_workspace/broker/` |
| Topic | named log | a folder of partitions |
| Partition | ordered, append-only log | a folder of segment files |
| Segment | `00000000000000000000.log` | same name, one JSON record per line |
| Offset | position in a partition | line number in the partition |
| Key → partition | murmur2 hash | crc32 hash |
| Consumer group offsets | `__consumer_offsets` topic | a JSON file per group |
| Replication, leader election, rebalancing protocol | yes | **not simulated** |

### OBJECTIVES
- See that a Kafka topic is just a set of append-only logs
- Explain how keys determine partitions, and why ordering is only guaranteed *within* a partition
- Track consumer progress with offsets, and explain consumer lag
- Show how consumer groups give both pub/sub (many groups) and work sharing (many members in one group)
- Use retention and replay, and explain what happens when data ages out
- Reproduce at-most-once, at-least-once, and effectively-once delivery
- Explain how Spark maps Kafka partitions to tasks and stores offsets in its checkpoint

---

### SETUP

Run on the PySpark 4.0 kernel. The Python Data Source API needs `pyarrow`.

All files this notebook creates live in `minikafka_workspace/`, which is deleted and recreated every time you run this cell.

In [ ]:
import os, json, glob, zlib, time, shutil, threading
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, get_json_object, window, count, countDistinct, avg, max as spark_max, expr
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType

try:
    import pyarrow
except ImportError:
    raise ImportError("The Python Data Source API needs pyarrow: run `pip install pyarrow` and restart the kernel")

spark = SparkSession.builder \
    .appName("MiniKafka-Demo") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "3") \
    .getOrCreate()

WORK_DIR = os.path.abspath("minikafka_workspace")
BROKER_DIR = os.path.join(WORK_DIR, "broker")
shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(BROKER_DIR)

print(f"Spark {spark.version}, broker at {BROKER_DIR}")

---

### 1. A Log is Just an Append-Only File

Before building anything Kafka-like, here is the core idea. A **log** is a file we only ever append to. We never update or delete a line in place.

Each event's position in the file is its **offset**. A reader doesn't need the writer's help to find its place: it only needs to remember the offset it reached.

In [ ]:
log_path = os.path.join(WORK_DIR, "my_first_log.jsonl")

events = [
    {"patient": "P-17", "event": "admitted"},
    {"patient": "P-17", "event": "dose: 5mg"},
    {"patient": "P-42", "event": "admitted"},
    {"patient": "P-17", "event": "dose: 5mg"},
    {"patient": "P-42", "event": "discharged"},
]

with open(log_path, "a") as f:
    for e in events:
        f.write(json.dumps(e) + "\n")

def read_from(path, start_offset):
    with open(path) as f:
        for offset, line in enumerate(f):
            if offset >= start_offset:
                yield offset, json.loads(line)

print("Reader A starts at the beginning:")
for offset, e in read_from(log_path, 0):
    print(" ", offset, e)

print("\nReader B already processed offsets 0-2, so it resumes at 3:")
for offset, e in read_from(log_path, 3):
    print(" ", offset, e)

**Point out:** the second dose of medication is a new event, not an update to a row. Both readers share the file, and neither one affects the other. This is the "events, not things" idea from the lecture.

---

### 2. The MiniKafka Broker: Topics, Partitions, and Segments

A **topic** is split into **partitions**. Each partition is its own log, so different machines can write and read different partitions at the same time.

Each partition's log is split into **segment** files. A segment is named after the first offset it holds. Splitting a partition into segments makes retention cheap: deleting old data means deleting whole old files.

```
broker/
  topics/transactions/
    meta.json
    partition-0/00000000000000000000.log   <- offsets 0-49
                00000000000000000050.log   <- offsets 50-99 (the "active" segment)
    partition-1/...
  consumer_offsets/
    fraud-detector/transactions.json       <- {"0": 112, "1": 98, "2": 105}
```

The cell below is the whole broker. Read it from top to bottom: it is short.

In [ ]:
SEGMENT_SIZE = 50   # records per segment file (real Kafka rolls segments by size, default 1 GB)

class OffsetOutOfRange(Exception):
    pass

# --- topics and partitions -------------------------------------------------

def topic_dir(topic):
    return os.path.join(BROKER_DIR, "topics", topic)

def partition_dir(topic, p):
    return os.path.join(topic_dir(topic), f"partition-{p}")

def create_topic(topic, partitions):
    if os.path.exists(topic_dir(topic)):
        raise ValueError(f"topic '{topic}' already exists")
    for p in range(partitions):
        os.makedirs(partition_dir(topic, p))
    with open(os.path.join(topic_dir(topic), "meta.json"), "w") as f:
        json.dump({"partitions": partitions}, f)

def num_partitions(topic):
    with open(os.path.join(topic_dir(topic), "meta.json")) as f:
        return json.load(f)["partitions"]

# --- segments and offsets --------------------------------------------------

def segments(topic, p):
    """Sorted list of (base_offset, path). A segment file is named after its first offset."""
    paths = glob.glob(os.path.join(partition_dir(topic, p), "*.log"))
    return sorted((int(os.path.basename(path)[:-4]), path) for path in paths)

def count_records(path):
    # only count complete lines, in case a producer is halfway through writing one
    with open(path) as f:
        return sum(1 for line in f if line.endswith("\n"))

def log_start_offset(topic, p):
    """Oldest offset still on disk (moves forward when retention deletes segments)."""
    segs = segments(topic, p)
    return segs[0][0] if segs else 0

def log_end_offset(topic, p):
    """The offset the next record will get."""
    segs = segments(topic, p)
    if not segs:
        return 0
    base, path = segs[-1]
    return base + count_records(path)

def start_offsets(topic):
    return {str(p): log_start_offset(topic, p) for p in range(num_partitions(topic))}

def end_offsets(topic):
    return {str(p): log_end_offset(topic, p) for p in range(num_partitions(topic))}

# --- writing and reading ---------------------------------------------------

def append(topic, p, records):
    """Append (key, value) records to partition p and return their offsets.
    Assumes one writer per partition (in real Kafka, the partition leader)."""
    next_offset = log_end_offset(topic, p)
    segs = segments(topic, p)
    base, path = segs[-1] if segs else (None, None)
    offsets = []
    for key, value in records:
        if path is None or next_offset - base >= SEGMENT_SIZE:   # roll a new segment
            base = next_offset
            path = os.path.join(partition_dir(topic, p), f"{base:020d}.log")
        record = {"offset": next_offset, "key": key, "value": value, "timestamp": int(time.time() * 1000)}
        with open(path, "a") as f:
            f.write(json.dumps(record) + "\n")
        offsets.append(next_offset)
        next_offset += 1
    return offsets

def read_partition(topic, p, start, end):
    """Yield the records with start <= offset < end."""
    if start < log_start_offset(topic, p):
        raise OffsetOutOfRange(f"{topic}-{p}: offset {start} was deleted by retention "
                               f"(log now starts at {log_start_offset(topic, p)})")
    segs = segments(topic, p)
    for i, (base, path) in enumerate(segs):
        next_base = segs[i + 1][0] if i + 1 < len(segs) else float("inf")
        if next_base <= start or base >= end:
            continue   # skip segments outside the requested range
        with open(path) as f:
            for line_no, line in enumerate(f):
                offset = base + line_no
                if offset >= end:
                    break
                if offset >= start:
                    yield json.loads(line)

# --- a look inside ---------------------------------------------------------

def show_broker():
    for root, dirs, files in os.walk(BROKER_DIR):
        dirs.sort(key=lambda d: (len(d), d))
        depth = os.path.relpath(root, BROKER_DIR).count(os.sep) + (root != BROKER_DIR)
        print("  " * depth + os.path.basename(root) + "/")
        for name in sorted(files):
            path = os.path.join(root, name)
            if name.endswith(".log"):
                detail = f"{count_records(path)} records"
            else:
                with open(path) as f:
                    detail = f.read().strip()
            print("  " * (depth + 1) + f"{name}  ({detail})")

In [ ]:
create_topic("transactions", partitions=3)
show_broker()

---

### 3. The Producer

A producer decides which partition each record goes to by hashing its **key**:

$$\text{partition} = \text{hash}(\text{key}) \bmod \text{number of partitions}$$

Real Kafka uses the murmur2 hash. We use crc32. Either works, as long as every producer computes the same answer for the same key. That is also why we *can't* use Python's built-in `hash()`: it is randomized per process, so two Spark workers would send the same key to different partitions.

In [ ]:
def partition_for(key, n):
    return zlib.crc32(key.encode()) % n

def produce(topic, key, value):
    p = partition_for(key, num_partitions(topic))
    offset = append(topic, p, [(key, value)])[0]
    return p, offset

for terminal in ["TERM-0", "TERM-1", "TERM-2", "TERM-0", "TERM-0"]:
    p, offset = produce("transactions", terminal, json.dumps({"terminal_id": terminal, "amount": 20.0}))
    print(f"{terminal} -> partition {p}, offset {offset}")

**Point out:** `TERM-0` always lands in the same partition, and its offsets keep counting up within that partition. Offsets are per partition, not per topic: partition 1 and partition 2 each have their own offset 0.

Now a more realistic workload. Our transactions are shaped like credit card swipes, and every 15th one is a fraud spike over $5,000.

In [ ]:
NUM_TERMINALS = 12
REGIONS = ["US-EAST", "EU-WEST", "APAC"]

def make_tx(i):
    """A simulated card transaction. Every 15th one is a $5,000+ fraud spike."""
    return {
        "tx_id": i,
        "terminal_id": f"TERM-{i % NUM_TERMINALS}",
        "region": REGIONS[i % 3],
        "amount": float(i % 100) + (5000.0 if i % 15 == 0 else 15.5),
        "tx_timestamp": datetime.now().isoformat(timespec="milliseconds"),
    }

def tx_dataframe(start, end):
    """Transactions start..end-1 as a DataFrame with Kafka's key/value columns."""
    rows = [(tx["terminal_id"], json.dumps(tx)) for tx in map(make_tx, range(start, end))]
    return spark.createDataFrame(rows, "key string, value string")

tx_dataframe(0, 3).show(truncate=False)

#### A Parallel Producer with Spark

To produce a lot of data, we let Spark do it in parallel. The trick is to shuffle rows so that **each Kafka partition is written by exactly one Spark task**. Then no two writers ever append to the same file.

In [ ]:
def spark_produce(df, topic):
    """Produce a DataFrame with `key` and `value` columns to a topic, one Spark task per partition."""
    n = num_partitions(topic)
    by_partition = df.select("key", "value").rdd \
        .map(lambda row: (partition_for(row.key, n), (row.key, row.value))) \
        .partitionBy(n, lambda p: p)   # Spark partition i holds exactly the rows for Kafka partition i

    def write_task(rows):
        rows = list(rows)
        if rows:
            append(topic, rows[0][0], [kv for _, kv in rows])

    by_partition.foreachPartition(write_task)
    print(f"produced {df.count()} records to '{topic}' using {by_partition.getNumPartitions()} parallel tasks")

spark_produce(tx_dataframe(0, 300), "transactions")
show_broker()

**Point out:** each partition now has several segment files of 50 records. Open one in a text editor: it's one JSON record per line, with its offset, key, value, and timestamp. That is all a Kafka log holds.

---

### 4. Reading a Topic with Spark

Next we teach Spark to read MiniKafka. With the Python Data Source API we implement two things:

- `partitions()`: split the work into pieces. Our pieces are *one Kafka partition between two offsets*, and Spark runs one task per piece.
- `read(piece)`: return the records in that piece.

Like Spark's real Kafka connector, our reader takes `startingOffsets` and `endingOffsets` options. They can be `"earliest"`, `"latest"`, or exact offsets per partition such as `{"0": 12, "1": 40, "2": 7}`.

In [ ]:
from pyspark.sql.datasource import DataSource, DataSourceReader, DataSourceStreamReader, InputPartition

RECORD_SCHEMA = "topic string, partition int, offset long, key string, value string, timestamp timestamp"

class TopicPartitionRange(InputPartition):
    """One unit of work: one topic partition between two offsets. Spark runs one task per range."""
    def __init__(self, topic, partition, start, end):
        self.topic, self.partition, self.start, self.end = topic, partition, start, end

def read_range(r):
    for rec in read_partition(r.topic, r.partition, r.start, r.end):
        yield (r.topic, r.partition, rec["offset"], rec["key"], rec["value"],
               datetime.fromtimestamp(rec["timestamp"] / 1000))

def resolve_offsets(spec, topic):
    if spec == "earliest":
        return start_offsets(topic)
    if spec == "latest":
        return end_offsets(topic)
    return {str(p): int(o) for p, o in json.loads(spec).items()}

class MiniKafkaBatchReader(DataSourceReader):
    def __init__(self, options):
        self.topic = options["subscribe"]
        self.start = resolve_offsets(options.get("startingoffsets", "earliest"), self.topic)
        self.end = resolve_offsets(options.get("endingoffsets", "latest"), self.topic)

    def partitions(self):
        return [TopicPartitionRange(self.topic, int(p), self.start[p], self.end[p])
                for p in sorted(self.start, key=int)]

    def read(self, partition):
        yield from read_range(partition)

class MiniKafkaDataSource(DataSource):
    @classmethod
    def name(cls):
        return "minikafka"

    def schema(self):
        return RECORD_SCHEMA

    def lowered_options(self):
        return {k.lower(): v for k, v in self.options.items()}

    def reader(self, schema):
        return MiniKafkaBatchReader(self.lowered_options())

spark.dataSource.register(MiniKafkaDataSource)

In [ ]:
all_tx = spark.read.format("minikafka").option("subscribe", "transactions").load()

print("Kafka partitions:", num_partitions("transactions"))
print("Spark partitions:", all_tx.rdd.getNumPartitions())
all_tx.show(5, truncate=60)

**Point out:** 3 Kafka partitions became 3 Spark partitions. This is exactly how Spark's real Kafka connector works, and it is why a topic's partition count caps how many Spark tasks can read it in parallel.

---

### 5. Keys, Partitioning, and Ordering

Kafka promises ordering **only within a partition**. Because a key always maps to the same partition, all events for one key stay in order. There is no order across partitions.

To see it, we produce an order's lifecycle (created → paid → shipped → delivered) for several orders, interleaved, from a single producer. `seq` records the global order in which events were sent.

In [ ]:
create_topic("order-events", partitions=3)

orders = [f"ORD-{n}" for n in range(108, 114)]
seq = 0
for status in ["created", "paid", "shipped", "delivered"]:
    for order in orders:
        produce("order-events", order, json.dumps({"order_id": order, "status": status, "seq": seq}))
        seq += 1

order_events = spark.read.format("minikafka").option("subscribe", "order-events").load() \
    .select("partition", "offset", "key",
            get_json_object("value", "$.status").alias("status"),
            get_json_object("value", "$.seq").cast("int").alias("seq"))

print("Every key lives in exactly one partition:")
order_events.groupBy("key").agg(countDistinct("partition").alias("partitions_used"),
                                count("*").alias("events")).orderBy("key").show()

In [ ]:
print("Within one key, offset order == send order:")
order_events.filter(col("key") == "ORD-108").orderBy("offset").show()

print("Reading the topic partition by partition does NOT give send order (look at seq):")
order_events.orderBy("partition", "offset").show(12)

**Point out:** if you need *every* event in one global order, you need a topic with 1 partition, and you give up parallelism. Choosing the key is choosing which events must stay in order (here: events for the same order).

---

### 6. Consumers and Offsets

A consumer reads a batch of records, processes them, then **commits** the offset it reached. The committed offset belongs to the consumer's **group**. If the consumer restarts, it resumes from its group's committed offset.

**Lag** is how far behind the latest offset a group is. It is the number one metric people watch in production Kafka.

In [ ]:
def offsets_file(group, topic):
    return os.path.join(BROKER_DIR, "consumer_offsets", group, f"{topic}.json")

def committed_offsets(group, topic):
    """Where the group resumes. A brand-new group starts at the earliest offset
    (Kafka's auto.offset.reset=earliest)."""
    offsets = start_offsets(topic)
    if os.path.exists(offsets_file(group, topic)):
        with open(offsets_file(group, topic)) as f:
            offsets.update(json.load(f))
    return offsets

def commit(group, topic, offsets):
    path = offsets_file(group, topic)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    current = {}
    if os.path.exists(path):
        with open(path) as f:
            current = json.load(f)
    current.update({str(p): int(o) for p, o in offsets.items()})
    with open(path + ".tmp", "w") as f:
        json.dump(current, f)
    os.replace(path + ".tmp", path)   # atomic: a crash never leaves a half-written offsets file

def lag(group, topic):
    committed, end = committed_offsets(group, topic), end_offsets(topic)
    return {p: end[p] - committed[p] for p in end}

def poll(group, topic, partitions=None):
    """Read everything from the group's committed offsets up to the current end.
    Returns the records and the offsets to commit once they are processed."""
    start, end = committed_offsets(group, topic), end_offsets(topic)
    if partitions is not None:
        start = {p: o for p, o in start.items() if int(p) in partitions}
        end = {p: end[p] for p in start}
    for p, o in start.items():
        if o < log_start_offset(topic, int(p)):
            raise OffsetOutOfRange(f"group '{group}' wants {topic}-{p} offset {o}, "
                                   f"but the log now starts at {log_start_offset(topic, int(p))}")
    df = spark.read.format("minikafka") \
        .option("subscribe", topic) \
        .option("startingOffsets", json.dumps(start)) \
        .option("endingOffsets", json.dumps(end)) \
        .load()
    return df, end

In [ ]:
tx_schema = StructType([
    StructField("tx_id", LongType()),
    StructField("terminal_id", StringType()),
    StructField("region", StringType()),
    StructField("amount", DoubleType()),
    StructField("tx_timestamp", TimestampType()),
])

def parse_tx(df):
    return df.select("partition", "offset", from_json("value", tx_schema).alias("tx")).select("partition", "offset", "tx.*")

df, end = poll("fraud-detector", "transactions")
print("records in this poll:", df.count())
parse_tx(df).filter("amount > 4000").show(5)

commit("fraud-detector", "transactions", end)
print("committed:", end)
print("lag:", lag("fraud-detector", "transactions"))

In [ ]:
df, end = poll("fraud-detector", "transactions")
print("poll again right away:", df.count(), "records")

spark_produce(tx_dataframe(300, 330), "transactions")
print("lag:", lag("fraud-detector", "transactions"))

df, end = poll("fraud-detector", "transactions")
print("poll after new data:", df.count(), "records")
commit("fraud-detector", "transactions", end)

**Point out:** the consumer only ever stores three integers. Kafka does no per-message bookkeeping, unlike a traditional message queue that tracks and deletes each message once it is acknowledged. That is a big part of why Kafka scales.

---

### 7. Consumer Groups

**Different groups** each get *every* record (pub/sub). A new `analytics` group starts from the beginning, no matter where `fraud-detector` is.

In [ ]:
df, end = poll("analytics", "transactions")
print("analytics sees", df.count(), "records")
parse_tx(df).groupBy("region").agg(count("*").alias("tx_count"), avg("amount").alias("avg_amount")).show()
commit("analytics", "transactions", end)

show_broker()

**Members of the same group** split the partitions between them (work sharing). Each partition goes to exactly one member, which keeps per-key ordering intact. When a member joins or leaves, the group **rebalances**.

Kafka has several assignment strategies. We use round robin.

In [ ]:
def assign_partitions(members, n):
    assignment = {m: [] for m in members}
    for p in range(n):
        assignment[members[p % len(members)]].append(p)
    return assignment

def run_group(group, topic, members):
    assignment = assign_partitions(members, num_partitions(topic))
    print(f"assignment: {assignment}")
    for member, parts in assignment.items():
        if not parts:
            print(f"  {member}: idle (no partitions left)")
            continue
        df, end = poll(group, topic, partitions=parts)
        print(f"  {member}: partitions {parts}, read {df.count()} records")
        commit(group, topic, end)

spark_produce(tx_dataframe(330, 390), "transactions")

print("Two workers in fraud-detector:")
run_group("fraud-detector", "transactions", ["worker-a", "worker-b"])

spark_produce(tx_dataframe(390, 450), "transactions")

print("\nworker-b crashed, rebalance:")
run_group("fraud-detector", "transactions", ["worker-a"])

spark_produce(tx_dataframe(450, 510), "transactions")

print("\nScale up to 4 workers:")
run_group("fraud-detector", "transactions", ["worker-a", "worker-b", "worker-c", "worker-d"])

**Point out:** with 3 partitions, a 4th worker sits idle. The partition count is the ceiling on parallelism within a group, so pick it with future scale in mind.

---

### 8. Retention and Replay

Because reading never deletes anything, a group can **replay** history just by moving its offsets back. That is how teams backfill a new model or reprocess after fixing a bug.

In [ ]:
def reset_offsets(group, topic, to="earliest"):
    commit(group, topic, resolve_offsets(to, topic))

print("analytics lag before reset:", lag("analytics", "transactions"))
reset_offsets("analytics", "transactions", "earliest")
print("analytics lag after reset: ", lag("analytics", "transactions"))

df, end = poll("analytics", "transactions")
print("replayed", df.count(), "records")
commit("analytics", "transactions", end)

Data isn't kept forever, though. **Retention** deletes the oldest segments (Kafka uses age, `retention.ms`, or size, `retention.bytes`). The active segment is never deleted.

If a group's committed offset points at deleted data, the consumer can't continue where it left off. In Kafka, `auto.offset.reset` decides what happens next: jump to the earliest data still available, jump to the latest, or fail.

In [ ]:
def apply_retention(topic, keep_segments):
    """Keep only the newest `keep_segments` segments of each partition."""
    for p in range(num_partitions(topic)):
        for base, path in segments(topic, p)[:-keep_segments]:
            os.remove(path)
            print(f"deleted {topic}-{p} segment starting at offset {base}")

# an audit job that last ran a long time ago, at the very start of the topic
commit("audit", "transactions", {"0": 0, "1": 0, "2": 0})

apply_retention("transactions", keep_segments=1)
print("log start offsets now:", start_offsets("transactions"))

try:
    poll("audit", "transactions")
except OffsetOutOfRange as e:
    print("\nOffsetOutOfRange:", e)

In [ ]:
reset_offsets("audit", "transactions", "earliest")   # auto.offset.reset=earliest
df, end = poll("audit", "transactions")
print(f"audit recovered, but can only read {df.count()} of the {sum(end_offsets('transactions').values())} records ever produced")
commit("audit", "transactions", end)

---

### 9. Delivery Semantics

Process first or commit first? The order of those two steps decides what happens when a consumer crashes in between.

| Order | Crash in between | Result |
|---|---|---|
| commit, then process | records committed but never processed | **at-most-once** (data lost) |
| process, then commit | records processed but not committed, so they are re-read | **at-least-once** (duplicates) |
| process, then commit, with an **idempotent** sink | duplicates are written but have no effect | **effectively-once** |

Our "processing" writes each record to a Parquet sink table. We simulate a crash between the two steps, restart the consumer, and count what reached the sink.

In [ ]:
create_topic("payments", partitions=3)
spark_produce(tx_dataframe(0, 100), "payments")

class SimulatedCrash(Exception):
    pass

def process_batch(df, sink_path):
    if not df.isEmpty():
        df.select("topic", "partition", "offset", "key", "value").write.mode("append").parquet(sink_path)

def consume(group, topic, sink_path, commit_timing, crash=False):
    df, end = poll(group, topic)
    if commit_timing == "before":
        commit(group, topic, end)
        if crash:
            raise SimulatedCrash("consumer died after committing, before processing")
        process_batch(df, sink_path)
    else:
        process_batch(df, sink_path)
        if crash:
            raise SimulatedCrash("consumer died after processing, before committing")
        commit(group, topic, end)

def sink_report(sink_path):
    if not glob.glob(os.path.join(sink_path, "*.parquet")):
        return "sink rows: 0 | distinct records: 0"
    sink = spark.read.parquet(sink_path)
    return f"sink rows: {sink.count()} | distinct records: {sink.select('partition', 'offset').distinct().count()}"

def run_scenario(name, commit_timing):
    sink_path = os.path.join(WORK_DIR, "sinks", name)
    try:
        consume(name, "payments", sink_path, commit_timing, crash=True)
    except SimulatedCrash as e:
        print(f"[{name}] CRASH: {e}")
    print(f"[{name}] restarting from committed offsets {committed_offsets(name, 'payments')}")
    consume(name, "payments", sink_path, commit_timing)
    print(f"[{name}] topic has 100 records | {sink_report(sink_path)}\n")
    return sink_path

run_scenario("at-most-once", commit_timing="before")
at_least_once_sink = run_scenario("at-least-once", commit_timing="after")

At-least-once is the usual choice: losing data is worse than seeing it twice. To get **effectively-once**, make the duplicates harmless. Each record already carries a unique ID, `(topic, partition, offset)`, so the sink can deduplicate on it. (In module 11, Delta Lake's `MERGE` does this kind of upsert for us.)

In [ ]:
deduped = spark.read.parquet(at_least_once_sink).dropDuplicates(["topic", "partition", "offset"])
print("effectively-once:", deduped.count(), "rows for 100 records")

**Point out:** Kafka producers face the same problem when they retry a send. That is why Kafka has idempotent producers (`enable.idempotence`) and transactions. Spark Structured Streaming solves the consumer side with the next section's checkpoint plus sinks that are idempotent or transactional.

---

### 10. Streaming: Spark Structured Streaming on MiniKafka

So far we polled by hand. Structured Streaming automates the loop: every trigger it asks the source for its latest offsets, reads the range since the last batch, and writes the offsets to its **checkpoint** *before* processing. The checkpoint plays the role of a consumer group's committed offsets.

A streaming source only needs a few more methods:

| Method | Spark asks | MiniKafka answers |
|---|---|---|
| `initialOffset()` | where does a brand-new query start? | earliest (or latest) |
| `latestOffset()` | what is the newest data? | each partition's log end offset |
| `partitions(start, end)` | split this batch into tasks | one range per partition |
| `read(partition)` | give me the rows | records in the range |
| `commit(end)` | batch done, clean up if you like | nothing to do |

In [ ]:
class MiniKafkaStreamReader(DataSourceStreamReader):
    def __init__(self, options):
        self.topic = options["subscribe"]
        self.starting = options.get("startingoffsets", "earliest")

    def initialOffset(self):
        # only used the first time a query runs; after that Spark resumes from its checkpoint
        return resolve_offsets(self.starting, self.topic)

    def latestOffset(self):
        return end_offsets(self.topic)

    def partitions(self, start, end):
        return [TopicPartitionRange(self.topic, int(p), start.get(p, 0), end[p])
                for p in sorted(end, key=int)]

    def read(self, partition):
        yield from read_range(partition)

    def commit(self, end):
        pass   # Spark has already recorded `end` in its checkpoint

class MiniKafkaStreamingDataSource(MiniKafkaDataSource):
    def streamReader(self, schema):
        return MiniKafkaStreamReader(self.lowered_options())

# replaces the batch-only "minikafka" source registered in section 4
spark.dataSource.register(MiniKafkaStreamingDataSource)

A background thread acts as a live producer, sending about 20 transactions per second. The streaming query computes the same sliding-window fraud dashboard as the earlier anomaly demo, but now it reads from a real (mini) Kafka topic.

In [ ]:
create_topic("live-transactions", partitions=3)

stop_producing = threading.Event()

def live_producer(topic, rate_per_sec=20):
    i = 0
    while not stop_producing.is_set():
        tx = make_tx(i)
        produce(topic, tx["terminal_id"], json.dumps(tx))
        i += 1
        time.sleep(1 / rate_per_sec)

producer_thread = threading.Thread(target=live_producer, args=("live-transactions",), daemon=True)
producer_thread.start()

stream = spark.readStream.format("minikafka").option("subscribe", "live-transactions").load()

dashboard = parse_tx(stream) \
    .groupBy(window(col("tx_timestamp"), "10 seconds", "5 seconds"), col("region")) \
    .agg(count("amount").alias("tx_count"),
         avg("amount").alias("average_spend"),
         spark_max("amount").alias("peak_transaction")) \
    .withColumn("FRAUD_ALERT", expr("peak_transaction > 4000.0")) \
    .sort(col("window.start").desc())

CHECKPOINT = os.path.join(WORK_DIR, "checkpoints", "fraud-dashboard")

def start_dashboard():
    return dashboard.writeStream \
        .format("console") \
        .outputMode("complete") \
        .option("truncate", "false") \
        .option("checkpointLocation", CHECKPOINT) \
        .trigger(processingTime="5 seconds") \
        .start()

query = start_dashboard()
query.awaitTermination(30)   # let it run for ~30 seconds
query.stop()
stop_producing.set()
producer_thread.join()

Spark reports, for every micro-batch, the offset range it read. These are exactly the dictionaries our `latestOffset()` returned:

In [ ]:
for progress in query.recentProgress[-3:]:
    source = progress["sources"][0]
    print(f"batch {progress['batchId']}: {source['startOffset']} -> {source['endOffset']}  ({source['numInputRows']} rows)")

latest_batch_file = max(glob.glob(os.path.join(CHECKPOINT, "offsets", "[0-9]*")), key=lambda f: int(os.path.basename(f)))
print(f"\nThe checkpoint's record of the last batch ({latest_batch_file}):")
with open(latest_batch_file) as f:
    print(f.read())

Now the payoff: data keeps arriving while the query is down. When we restart it with the same checkpoint, it resumes from where it stopped instead of starting over, just like a consumer group.

In [ ]:
print("end offsets when the query stopped:", end_offsets("live-transactions"))
for i in range(1000, 1060):
    tx = make_tx(i)
    produce("live-transactions", tx["terminal_id"], json.dumps(tx))
print("end offsets after more data arrived:", end_offsets("live-transactions"))

query = start_dashboard()
query.awaitTermination(10)
query.stop()

first = query.recentProgress[0]["sources"][0]
print(f"\nrestarted query's first batch read {first['startOffset']} -> {first['endOffset']}")

---

### 11. Swapping in Real Kafka

Every piece of MiniKafka has a real Kafka equivalent:

| MiniKafka | Real Kafka |
|---|---|
| `create_topic("t", partitions=3)` | `kafka-topics.sh --create --topic t --partitions 3 --replication-factor 3` |
| `produce(topic, key, value)` | `KafkaProducer.send(topic, key=..., value=...)` |
| `spark_produce(df, topic)` | `df.write.format("kafka").option("topic", topic).save()` |
| `poll()` / `commit()` | `KafkaConsumer.poll()` / `commit()` |
| `consumer_offsets/<group>/<topic>.json` | the internal `__consumer_offsets` topic |
| `apply_retention()` | the broker's log cleaner, driven by `retention.ms` / `retention.bytes` |
| crc32 partitioner | murmur2 partitioner |

And the streaming query barely changes. The real source's offset options nest by topic, and it returns `key` and `value` as binary, so you cast them:

```python
stream = spark.readStream.format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "live-transactions") \
    .option("startingOffsets", "earliest") \
    .load() \
    .selectExpr("topic", "partition", "offset", "CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value", "timestamp")
```

(this needs the `org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0` package and a running broker)

What MiniKafka leaves out: the network, **replication** (each partition copied to several brokers, with one leader taking writes), leader election when a broker dies, and the group coordinator that runs rebalances.

---

### TRY IT

1. Recreate `transactions` with 1 partition and rerun section 5. What happens to ordering? To parallelism?
2. Key the transactions by `region` instead of `terminal_id`. How evenly are the 3 partitions loaded? Why is a key with few distinct values risky?
3. Replace `apply_retention` with a time-based version that deletes segments whose newest record is older than N seconds.
4. Simulate replication: write each record to `partition-0` and to a `partition-0-replica` folder. When is it safe to tell the producer the write succeeded? (look up `acks=all`)
5. In section 10, crash the streaming query mid-run (stop the kernel), restart it, and check the checkpoint. Which delivery semantics does Spark give you with the console sink?

In [ ]:
spark.stop()